### Imports and config 

In [52]:
import os
import requests
import pandas as pd

### Load Paths and CSV's 

In [53]:
'''# ----------- Konfiguration -----------
PDB_URL = "https://opig.stats.ox.ac.uk/webapps/sabdab-sabpred/sabdab/pdb/{}/?scheme=chothia"
PDB_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "data", "pdbs_cnf"))  # ../data/pdbs_cnf
#PDB_DIR = os.path.join(os.path.dirname(os.getcwd()), "data")  # Put to folder: ../data/pdbs_cnf
CSV_FILENAME = "pdb_ids_sars_cov2.csv"  # or "pdb_ids.csv"

# ----------- Preperation -----------
# Hier KEIN os.makedirs(), weil der Ordner "data" laut dir schon existiert

csv_path = os.path.join(os.path.dirname(__file__) if '__file__' in globals() else os.getcwd(), CSV_FILENAME)
df = pd.read_csv(csv_path, header=None)

# Nur gültige 4-stellige PDB-IDs behalten
unique_pdb_ids = df[0].dropna().astype(str).str.strip()
unique_pdb_ids = unique_pdb_ids[unique_pdb_ids.str.match(r"^[A-Za-z0-9]{4}$")].tolist()

print(f"{len(unique_pdb_ids)} gültige PDB-IDs geladen.")'''

'# ----------- Konfiguration -----------\nPDB_URL = "https://opig.stats.ox.ac.uk/webapps/sabdab-sabpred/sabdab/pdb/{}/?scheme=chothia"\nPDB_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "data", "pdbs_cnf"))  # ../data/pdbs_cnf\n#PDB_DIR = os.path.join(os.path.dirname(os.getcwd()), "data")  # Put to folder: ../data/pdbs_cnf\nCSV_FILENAME = "pdb_ids_sars_cov2.csv"  # or "pdb_ids.csv"\n\n# ----------- Preperation -----------\n# Hier KEIN os.makedirs(), weil der Ordner "data" laut dir schon existiert\n\ncsv_path = os.path.join(os.path.dirname(__file__) if \'__file__\' in globals() else os.getcwd(), CSV_FILENAME)\ndf = pd.read_csv(csv_path, header=None)\n\n# Nur gültige 4-stellige PDB-IDs behalten\nunique_pdb_ids = df[0].dropna().astype(str).str.strip()\nunique_pdb_ids = unique_pdb_ids[unique_pdb_ids.str.match(r"^[A-Za-z0-9]{4}$")].tolist()\n\nprint(f"{len(unique_pdb_ids)} gültige PDB-IDs geladen.")'

### Create folder for PDB files, load paths and CSV's
exact the same function as above but now the folder gets automatically created and must not be manually created from each user

In [54]:


# ----------- Konfiguration -----------
PDB_URL = "https://opig.stats.ox.ac.uk/webapps/sabdab-sabpred/sabdab/pdb/{}/?scheme=chothia"
PDB_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "data", "pdbs_corona_cnf"))  # ../data/pdbs_corona_cnf
CSV_FILENAME = "pdb_ids_sars_cov2.csv"  # or "pdb_ids.csv"

# ----------- Vorbereitung -----------
# Ordner erstellen, falls er noch nicht existiert
os.makedirs(PDB_DIR, exist_ok=True)

# Pfad zur CSV-Datei im aktuellen Arbeitsverzeichnis oder Notebook-Verzeichnis
csv_path = os.path.join(os.path.dirname(__file__) if '__file__' in globals() else os.getcwd(), CSV_FILENAME)

# CSV-Datei einlesen
df = pd.read_csv(csv_path, header=None)

# Nur gültige 4-stellige PDB-IDs behalten
unique_pdb_ids = df[0].dropna().astype(str).str.strip()
unique_pdb_ids = unique_pdb_ids[unique_pdb_ids.str.match(r"^[A-Za-z0-9]{4}$")].tolist()

print(f"{len(unique_pdb_ids)} gültige PDB-IDs geladen.")
print(f"Zieldateien werden gespeichert in: {PDB_DIR}")

379 gültige PDB-IDs geladen.
Zieldateien werden gespeichert in: /Users/loa/Documents/GitHub/group04-team02/data/pdbs_corona_cnf


### Download function

In [55]:
'''def download_pdb(pdb_id):
    url = PDB_URL.format(pdb_id)
    filename = os.path.join(PDB_DIR, f"{pdb_id}.pdb")

    if os.path.exists(filename):
        print(f"{pdb_id} already exists.")
        return filename

    response = requests.get(url)
    if response.status_code == 200 and len(response.content) > 100:
        with open(filename, "wb") as f:
            f.write(response.content)
        print(f"{pdb_id} downloaded successfully.")
        return filename
    else:
        print(f"{pdb_id} could not be downloaded.")
        return None'''

'def download_pdb(pdb_id):\n    url = PDB_URL.format(pdb_id)\n    filename = os.path.join(PDB_DIR, f"{pdb_id}.pdb")\n\n    if os.path.exists(filename):\n        print(f"{pdb_id} already exists.")\n        return filename\n\n    response = requests.get(url)\n    if response.status_code == 200 and len(response.content) > 100:\n        with open(filename, "wb") as f:\n            f.write(response.content)\n        print(f"{pdb_id} downloaded successfully.")\n        return filename\n    else:\n        print(f"{pdb_id} could not be downloaded.")\n        return None'

In [56]:
def download_pdb(pdb_id):
    filename = os.path.join(PDB_DIR, f"{pdb_id}.pdb")

    # Wenn Datei schon existiert und sinnvoll groß ist → überspringen
    if os.path.exists(filename):
        if os.path.getsize(filename) > 100:  # schützt auch vor defekten Dateien
            print(f"{pdb_id} already exists – skipped.")
            return filename
        else:
            print(f"{pdb_id} exists but is too small – redownloading.")

    # Herunterladen
    url = PDB_URL.format(pdb_id)
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200 and len(response.content) > 100:
            with open(filename, "wb") as f:
                f.write(response.content)
            print(f"{pdb_id} downloaded successfully.")
            return filename
        else:
            print(f"{pdb_id} could not be downloaded (invalid content or status code).")
            return None
    except requests.exceptions.RequestException as e:
        print(f"{pdb_id} download failed due to connection error: {e}")
        return None


### Start downloads

In [57]:
print(f"Starte Download von {len(unique_pdb_ids)} PDB-Dateien...\n")

for i, pdb_id in enumerate(unique_pdb_ids, 1):
    print(f"[{i}/{len(unique_pdb_ids)}] {pdb_id}:", end=" ")
    download_pdb(pdb_id)

print("\nFertig.")

Starte Download von 379 PDB-Dateien...

[1/379] 9cci: 9cci downloaded successfully.
[2/379] 9ccj: 9ccj downloaded successfully.
[3/379] 9bj2: 9bj2 downloaded successfully.
[4/379] 8z6r: 8z6r downloaded successfully.
[5/379] 8z6s: 8z6s downloaded successfully.
[6/379] 8z6t: 8z6t downloaded successfully.
[7/379] 8z6x: 8z6x downloaded successfully.
[8/379] 8v5v: 8v5v downloaded successfully.
[9/379] 9c44: 9c44 downloaded successfully.
[10/379] 8s6m: 8s6m downloaded successfully.
[11/379] 8wpw: 8wpw downloaded successfully.
[12/379] 9atm: 9atm downloaded successfully.
[13/379] 9au1: 9au1 downloaded successfully.
[14/379] 8kdm: 8kdm downloaded successfully.
[15/379] 8keo: 8keo downloaded successfully.
[16/379] 8kep: 8kep downloaded successfully.
[17/379] 8ker: 8ker downloaded successfully.
[18/379] 8sdh: 8sdh downloaded successfully.
[19/379] 8vcr: 8vcr downloaded successfully.
[20/379] 8yww: 8yww downloaded successfully.
[21/379] 8q5y: 8q5y downloaded successfully.
[22/379] 8xse: 8xse down